## SRP162630 / GSE120493

**paper:** [https://doi.org/10.1101/377358](https://www.biorxiv.org/content/10.1101/377358v1.full) - Subdivision of ancestral scale genetic program underlies origin of feathers and avian scutate scales, 2018

**date, curator:** 2026-03-11, Sara Carsanaro

**notes**
* it seems like the chickens might be White Leghorn, although it is not clear if there are also some Silkies
* need to do dev stages manually since chicken needs its own ontology
    * alligator [ferguson dev stages](https://www.fao.org/4/t0226e/t0226e06.htm) explained
* not clear which library prep kit used, but it is polyA

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,asymmetric (femoral) scale morphogenesis tract epidermis,UBERON:0008201,scute,missing child term
2,asymmetric (femoral) scale placode tract epidermis,UBERON:0011817,skin appendage placode,missing child term
4,claw morphogenesis dermis,UBERON:0005418,hindlimb bud,missing child term
5,claw morphogenesis epidermis,UBERON:0005418,hindlimb bud,missing child term
9,claw placode dermis,UBERON:0005733,limb field,missing child term
10,claw placode epidermis,UBERON:0005733,limb field,missing child term
13,feather morphogenesis dorsal tract dermis,UBERON:0011803,"feather bud, dermal component",missing child term
14,feather morphogenesis dorsal tract epidermis,UBERON:0011804,"feather bud, epidermal component",missing child term
18,feather morphogenesis femoral tract epidermis,UBERON:0011804,"feather bud, epidermal component",missing child term
20,feather morphogenesis flank/femoral tract epidermis,UBERON:0011804,"feather bud, epidermal component",missing child term


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Feguson stage 23/24,UBERON:0000111,organogenesis stage,other
2,Ferguson stage 21,UBERON:0000111,organogenesis stage,other
4,Hamburger and Hamilton stage 38,GgalDv:0000052,Hamburger Hamilton stage 38,perfect match
7,Hamburger and Hamilton stage 38,UBERON:0000111,organogenesis stage,other
9,Hamburger and Hamilton Stage 36,GgalDv:0000050,Hamburger Hamilton stage 36,perfect match
12,Hamburger and Hamilton Stage 36,UBERON:0000111,organogenesis stage,other
23,Hamburger and Hamilton Stage 33/34,GgalDv:0000094,organogenesis stage,other
26,Hamburger and Hamilton Stage 33/34,UBERON:0000111,organogenesis stage,other
33,Ferguson stage 22,UBERON:0000111,organogenesis stage,other
37,Hamburger and Hamilton stage 40,GgalDv:0000054,Hamburger Hamilton stage 40,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP162630"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [6]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,,other,,,,8496,,,,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,,other,,,,8496,,,,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,,other,,,,8496,,,,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,,other,,,,8496,,,,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,,perfect match,,,,9031,,,,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,claw morphogenesis dermis,,Hamburger and Hamilton stage 38,,TRANSCRIPTOMIC,cDNA
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,missing child term,,perfect match,,,,9031,,,,,,C56C,"SAMN10131484,GSM3401685",,,,,,

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['asymmetric (femoral) scale morphogenesis tract epidermis'
 'asymmetric (femoral) scale placode tract epidermis'
 'claw morphogenesis dermis' 'claw morphogenesis epidermis'
 'claw placode dermis' 'claw placode epidermis'
 'feather morphogenesis dorsal tract dermis'
 'feather morphogenesis dorsal tract epidermis'
 'feather morphogenesis femoral tract epidermis'
 'feather morphogenesis flank/femoral tract epidermis'
 'feather morpogenesis femoral tract dermis'
 'feather placode dorsal tract dermis'
 'feather placode dorsal tract epidermis'
 'feather placode femoral tract dermis'
 'feather placode femoral tract epidermis'
 'feather placode flank/femoral tract epidermis'
 'foot claw morphogenesis epidermis' 'foot claw placode epidermis'
 'reticulate scale morphogenesis tract dermis'
 'reticulate scale morphogenesis tract epidermis'
 'reticulate scale placode tract dermis'
 'reticulate scale placode tract epidermis'
 'scutate scale morphogenesis tact epidermis'
 'scutate scale morphogenesi

some thoughts on the different scales and feathers - annotating manually:

scales
* asymmetric (femoral), all epidermis, placode and morphogensis - alligator only
* symmetric (flank), all epidermis, placode and morphogensis - alligator only
* reticulate, dermis and epidermis, placode and morphogensis - chicken and emu only
    * these are round scales, found on the soles of feet
* scutate, dermis and epidermis, placode and morphogensis - chicken and emu only
    * these are rectanguar scales, found on the tops of feet
* uberon doesn't have great scale options for non-fish
    * possibly UBERON:0008201 scute for morphogensis
    * UBERON:0011817 skin appendage placode for placode ones

feather
* dorsal tract, dermis and epidermis, placode and morphogensis - chicken and emu
* femoral tract, dermis and epidermis, placode and morphogensis - chicken only
* flank/femoral, epidermis only, placode and morphogensis - emu only
* i don't think i can use uberon to mark dorsal or femoral/flank, instead to use placode and bud (morphogenesis)
    * UBERON:0011783 feather follicle placode
    * UBERON:0011803 feather bud, dermal component
    * UBERON:0011804 feather bud, epidermal component

claw
* dermis and epidermis, placode and morphogensis - chicken and emu only
* foot claw, epidermis only, placode and morphogensis - aligator only
* i think we can use hindlimb bud for both or limb field for placode
    * UBERON:0005418 hindlimb bud - morphogensis 
    * UBERON:0005733 limb field - placode

In [8]:

# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,,,,8496,,,,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,,,,8496,,,,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,,,,8496,,,,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,,,,8496,,,,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,,,,9031,,,,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,claw morphogenesis dermis,,Hamburger and Hamilton stage 38,,TRANSCRIPTOMIC,cDNA
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,missing child

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [9]:
unique_sorted(library, "infoStage")

['Feguson stage 23/24' 'Ferguson stage 21' 'Ferguson stage 22'
 'Ferguson stage 23' 'Hamburger and Hamilton Stage 33/34'
 'Hamburger and Hamilton Stage 36' 'Hamburger and Hamilton Stage 37'
 'Hamburger and Hamilton Stage 39' 'Hamburger and Hamilton stage 38'
 'Hamburger and Hamilton stage 40']


In [ ]:
# all
#library.loc[:,'stageId'] = ''
#library.loc[:,'stageName'] = ''
# perfect match, missing child term, other
#library.loc[:,'stageAnnotationStatus'] = ''

# view
display_df(library)

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [10]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,claw morphogenesis dermis,,Hamburger and Hamilton stage 38,,TRANSCRIPTOMIC,cDNA
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,mis

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [11]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,,,,,11/03/2026,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,claw morphogenesis dermis,,Hamburger and Hamilton stage 38,,TRANSCRIPTOMIC,cDNA
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epidermis,Hamburger 

#### globin, replicates

In [12]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-16'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,,,,SAC,2026-03-16,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,,,,SAC,2026-03-16,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale morphogenesis tract epidermis,,Feguson stage 23/24,,TRANSCRIPTOMIC,cDNA
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,,,,SAC,2026-03-16,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,,,,SAC,2026-03-16,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,asymmetric (femoral) scale placode tract epidermis,,Ferguson stage 21,,TRANSCRIPTOMIC,cDNA
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,,,,SAC,2026-03-16,Total RNA was extracted using a Qiagen Rneasy kit. polyA enriched strand-specific library was constructed using a proprietary method developed by the Yale Center for Genome Analysis,,,,claw morphogenesis dermis,,Hamburger and Hamilton stage 38,,TRANSCRIPTOMIC,cDNA
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epide

#### comments

In [14]:
library.loc[:,'comment'] = 'no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full'

#### save complete file with correct columns

In [15]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401700,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
1,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401699,asymmetric (femoral) scale morphogenesis tract epidermis,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
2,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401694,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
3,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401693,asymmetric (femoral) scale placode tract epidermis,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
4,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401690,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
5,SRX4740521,SRP162630,Illumina HiSeq 2000,SRS3821852,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401685,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C56C,"SAMN10131484,GSM3401685",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
6,SRX4740520,SRP162630,Illumina HiSeq 2000,SRS3821851,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401684,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C49C,"SAMN10131485,GSM3401684",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
7,SRX4740556,SRP162630,Illumina HiSeq 2000,SRS3821886,UBERON:0005418,hindlimb bud,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM3401720,claw morphogenesis epidermis,Hamburger and Hamilton stage 38,missing child term,not documented,other,NA,,,8790,,,polyA,,,E16C,"SAMN10131511,GSM3401720",,,,,,,"no pmid, https://www.biorxiv.org/content/10.1101/377358v1.full",,,SAC,2026-03-16
8,SRX4740555,SRP162630,Illumina HiSeq 2000,SRS3821884,UBERON:0005418,hindlimb bud,UBERON:0000111,organogenesis stage,https://www

### experiment annotations

In [16]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP162630,Subdivision of ancestral scale genetic program underlies origin of feathers and avian scutate scales,"Birds and other reptiles possess a diversity of feather and scale-like skin appendages. Feathers are commonly assumed to have originated from ancestral scales in theropod dinosaurs. However, most birds also have scaled feet, indicating birds evolved the capacity to grow both ancestral and derived morphologies. This suggests a more complex evolutionary history than a simple linear transition between feathers and scales. We set out to investigate the evolution of feathers via the comparison of transcriptomes assembled from diverse skin appendages in chicken, emu, and alligator. Our data reveal that feathers and the overlapping ‘scutate’ scales of birds share more similar gene expression to each other, and to two types of alligator scales, than they do to the tuberculate ‘reticulate’ scales on bird footpads. Accordingly, we propose a history of skin appendage diversification, in which feathers and bird scutate scales arose from ancestral archosaur body scales, whereas reticulate scales arose earlier in tetrapod evolution. We also show that many “feather-specific genes” are also expressed in alligator scales. In-situ hybridization results in feather buds suggest that these genes represent ancestral scale genes that acquired novel roles in feather morphogenesis and were repressed in bird scales. Our findings suggest that the differential reuse, in feathers, and suppression, in bird scales, of genes ancestrally expressed in archosaur scales has been a key factor in the origin of feathers – and may represent an important mechanism for the origin of evolutionary novelties. Overall design: In chicken and emu we sampled two biological replicates for epidermis from each of five skin tracts at two developmental stages (placode and morphogenesis): dorsal and femoral/flank feathers, scutate scales, reticulate scales, and claws. In chicken, we additionally sampled a single biological replicate from dermis from all 5 skin tracts at both developmental stages. In alligator we sampled two biological replicates for epidermis from 3 skin tracts at two developmental stages (placode and morphogenesis): dorsal scales, femoral scales, and claws.",SRA,,,,,,GSE120493,PRJNA493197,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [17]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

60

In [18]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP162630,Subdivision of ancestral scale genetic program underlies origin of feathers and avian scutate scales,"Birds and other reptiles possess a diversity of feather and scale-like skin appendages. Feathers are commonly assumed to have originated from ancestral scales in theropod dinosaurs. However, most birds also have scaled feet, indicating birds evolved the capacity to grow both ancestral and derived morphologies. This suggests a more complex evolutionary history than a simple linear transition between feathers and scales. We set out to investigate the evolution of feathers via the comparison of transcriptomes assembled from diverse skin appendages in chicken, emu, and alligator. Our data reveal that feathers and the overlapping ‘scutate’ scales of birds share more similar gene expression to each other, and to two types of alligator scales, than they do to the tuberculate ‘reticulate’ scales on bird footpads. Accordingly, we propose a history of skin appendage diversification, in which feathers and bird scutate scales arose from ancestral archosaur body scales, whereas reticulate scales arose earlier in tetrapod evolution. We also show that many “feather-specific genes” are also expressed in alligator scales. In-situ hybridization results in feather buds suggest that these genes represent ancestral scale genes that acquired novel roles in feather morphogenesis and were repressed in bird scales. Our findings suggest that the differential reuse, in feathers, and suppression, in bird scales, of genes ancestrally expressed in archosaur scales has been a key factor in the origin of feathers – and may represent an important mechanism for the origin of evolutionary novelties. Overall design: In chicken and emu we sampled two biological replicates for epidermis from each of five skin tracts at two developmental stages (placode and morphogenesis): dorsal and femoral/flank feathers, scutate scales, reticulate scales, and claws. In chicken, we additionally sampled a single biological replicate from dermis from all 5 skin tracts at both developmental stages. In alligator we sampled two biological replicates for epidermis from 3 skin tracts at two developmental stages (placode and morphogenesis): dorsal scales, femoral scales, and claws.",SRA,total,Bgee 1K,60,,,GSE120493,PRJNA493197,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [19]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://www.biorxiv.org/content/10.1101/377358v1.full'
experiment.loc[:,'DOI'] = '10.1101/377358'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP162630,Subdivision of ancestral scale genetic program underlies origin of feathers and avian scutate scales,"Birds and other reptiles possess a diversity of feather and scale-like skin appendages. Feathers are commonly assumed to have originated from ancestral scales in theropod dinosaurs. However, most birds also have scaled feet, indicating birds evolved the capacity to grow both ancestral and derived morphologies. This suggests a more complex evolutionary history than a simple linear transition between feathers and scales. We set out to investigate the evolution of feathers via the comparison of transcriptomes assembled from diverse skin appendages in chicken, emu, and alligator. Our data reveal that feathers and the overlapping ‘scutate’ scales of birds share more similar gene expression to each other, and to two types of alligator scales, than they do to the tuberculate ‘reticulate’ scales on bird footpads. Accordingly, we propose a history of skin appendage diversification, in which feathers and bird scutate scales arose from ancestral archosaur body scales, whereas reticulate scales arose earlier in tetrapod evolution. We also show that many “feather-specific genes” are also expressed in alligator scales. In-situ hybridization results in feather buds suggest that these genes represent ancestral scale genes that acquired novel roles in feather morphogenesis and were repressed in bird scales. Our findings suggest that the differential reuse, in feathers, and suppression, in bird scales, of genes ancestrally expressed in archosaur scales has been a key factor in the origin of feathers – and may represent an important mechanism for the origin of evolutionary novelties. Overall design: In chicken and emu we sampled two biological replicates for epidermis from each of five skin tracts at two developmental stages (placode and morphogenesis): dorsal and femoral/flank feathers, scutate scales, reticulate scales, and claws. In chicken, we additionally sampled a single biological replicate from dermis from all 5 skin tracts at both developmental stages. In alligator we sampled two biological replicates for epidermis from 3 skin tracts at two developmental stages (placode and morphogenesis): dorsal scales, femoral scales, and claws.",SRA,total,Bgee 1K,60,,,GSE120493,PRJNA493197,,https://www.biorxiv.org/content/10.1101/377358v1.full,10.1101/377358,,


#### comments

In [20]:
experiment.loc[:,'comment'] = 'no pmid'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP162630,Subdivision of ancestral scale genetic program underlies origin of feathers and avian scutate scales,"Birds and other reptiles possess a diversity of feather and scale-like skin appendages. Feathers are commonly assumed to have originated from ancestral scales in theropod dinosaurs. However, most birds also have scaled feet, indicating birds evolved the capacity to grow both ancestral and derived morphologies. This suggests a more complex evolutionary history than a simple linear transition between feathers and scales. We set out to investigate the evolution of feathers via the comparison of transcriptomes assembled from diverse skin appendages in chicken, emu, and alligator. Our data reveal that feathers and the overlapping ‘scutate’ scales of birds share more similar gene expression to each other, and to two types of alligator scales, than they do to the tuberculate ‘reticulate’ scales on bird footpads. Accordingly, we propose a history of skin appendage diversification, in which feathers and bird scutate scales arose from ancestral archosaur body scales, whereas reticulate scales arose earlier in tetrapod evolution. We also show that many “feather-specific genes” are also expressed in alligator scales. In-situ hybridization results in feather buds suggest that these genes represent ancestral scale genes that acquired novel roles in feather morphogenesis and were repressed in bird scales. Our findings suggest that the differential reuse, in feathers, and suppression, in bird scales, of genes ancestrally expressed in archosaur scales has been a key factor in the origin of feathers – and may represent an important mechanism for the origin of evolutionary novelties. Overall design: In chicken and emu we sampled two biological replicates for epidermis from each of five skin tracts at two developmental stages (placode and morphogenesis): dorsal and femoral/flank feathers, scutate scales, reticulate scales, and claws. In chicken, we additionally sampled a single biological replicate from dermis from all 5 skin tracts at both developmental stages. In alligator we sampled two biological replicates for epidermis from 3 skin tracts at two developmental stages (placode and morphogenesis): dorsal scales, femoral scales, and claws.",SRA,total,Bgee 1K,60,,,GSE120493,PRJNA493197,,https://www.biorxiv.org/content/10.1101/377358v1.full,10.1101/377358,,no pmid


#### save complete file

In [21]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [22]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [25]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [26]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56230,SRX4377511,SRP152991,Illumina HiSeq 2500,SRS3533660,UBERON:6001778,insect wing disc,UBERON:0014405,nymph stage,,wing pads,fifth (final) instar nymph,other,not documented,missing child term,NA,NA,,197043,KAPA Stranded RNA-Seq kit,full_length,polyA,,,Invertebrate sample from Homalodisca vitripennis,SAMN07822717,,,,,,,[Bgee curator notes: see https://github.com/ob...,,,ANN,2026-03-17
56231,SRX4377510,SRP152991,Illumina HiSeq 2500,SRS3533661,UBERON:8450007,insect mesothoracic leg,UBERON:0014405,nymph stage,,T2 legs,fifth (final) instar nymph,perfect match,not documented,missing child term,NA,NA,,1464891,KAPA Stranded RNA-Seq kit,full_length,polyA,,,Invertebrate sample from Entylia carinata (bro...,SAMN07822619,,,,,,,,,,ANN,2026-03-17
56232,SRX4740536,SRP162630,Illumina HiSeq 2000,SRS3821866,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,asymmetric (femoral) scale morphogenesis tract...,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A190Fem,"SAMN10131469,GSM3401700",,,,,,,"no pmid, https://www.biorxiv.org/content/10.11...",,,SAC,2026-03-16
56233,SRX4740535,SRP162630,Illumina HiSeq 2000,SRS3821865,UBERON:0008201,scute,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,asymmetric (femoral) scale morphogenesis tract...,Feguson stage 23/24,missing child term,not documented,other,NA,,,8496,,,polyA,,,A189Fem,"SAMN10131470,GSM3401699",,,,,,,"no pmid, https://www.biorxiv.org/content/10.11...",,,SAC,2026-03-16
56234,SRX4740530,SRP162630,Illumina HiSeq 2000,SRS3821860,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,asymmetric (femoral) scale placode tract epide...,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A83Fem,"SAMN10131478,GSM3401694",,,,,,,"no pmid, https://www.biorxiv.org/content/10.11...",,,SAC,2026-03-16
56235,SRX4740529,SRP162630,Illumina HiSeq 2000,SRS3821870,UBERON:0011817,skin appendage placode,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,asymmetric (femoral) scale placode tract epide...,Ferguson stage 21,missing child term,not documented,other,NA,,,8496,,,polyA,,,A82Fem,"SAMN10131479,GSM3401693",,,,,,,"no pmid, https://www.biorxiv.org/content/10.11...",,,SAC,2026-03-16
56236,SRX4740526,SRP162630,Illumina HiSeq 2000,SRS3821857,UBERON:0005418,hindlimb bud,GgalDv:0000052,Hamburger Hamilton stage 38,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,claw morphogenesis dermis,Hamburger and Hamilton stage 38,missing child term,not documented,perfect match,NA,,,9031,,,polyA,,,C56Cd,"SAMN10131474,GSM3401690",,,,,,,"no pmid, https://www.biorxiv.org/content/10.11...",,,SAC,2026-03-16


In [27]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1090,SRP165825,Five genomes of Bird-of-Paradise Genome sequen...,We sequenced five genomes of birds-of-paradise...,SRA,partial,Bgee 1K,2,,,,PRJNA491255,30936435,https://www.nature.com/articles/s41559-019-0850-1,10.1038/s41559-019-0850-1,,removed DNA seq libraries
1091,SRP152991,Comparative tissue specific transcriptomes of ...,We used RNA-Seq to test hypotheses for the ori...,SRA,total,Bgee 1K,63,KAPA Stranded RNA-Seq kit,full_length,,PRJNA415461,31819237,https://www.nature.com/articles/s41559-019-1054-4,10.1038/s41559-019-1054-4,,"Requested 3 NTR in UBERON, see https://github...."
1092,SRP162630,Subdivision of ancestral scale genetic program...,Birds and other reptiles possess a diversity o...,SRA,total,Bgee 1K,60,,,GSE120493,PRJNA493197,,https://www.biorxiv.org/content/10.1101/377358...,10.1101/377358,,no pmid


### add annotations to git

In [28]:
! git pull

Already up to date.


In [29]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [30]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
! git add $git_experiment_path $git_library_path

In [32]:
! git commit -m $commit_message_exp

[develop 4a7de3f] adding annotated bulk experiment SRP162630
 2 files changed, 61 insertions(+)


In [33]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.85 KiB | 1.21 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   c8293ad..4a7de3f  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push